# TE-292 — Aula 3: juntando dados de manutenção

## Etapa B2 — Uso operacional e integração por ativo (versão revisada)

**Tempo previsto:** 60 minutos

Esta é uma revisão de `B_usage_data.ipynb` feita após uma checagem de
qualidade da fonte `usage_data.csv` e, seguindo a mesma lógica de
`A2_maint_part_cons.ipynb`, aproveitando uma tabela de consumo mais rica que
a versão original de B nem chegava a usar. A etapa 1 (visão geral) é igual à
versão original. A partir da etapa 2 incorporamos seis melhorias encontradas
na checagem:

1. **duplicatas exatas** em `usage_data.csv` não eram detectadas nem tratadas;
2. **leituras conflitantes** — mesmo `Asset` e mesmo `Time`, valor de `Use`
   diferente — não eram sinalizadas;
3. **reversões de leitura** (`Use` menor que a leitura anterior do mesmo
   ativo) não eram diagnosticadas, embora a fórmula `nb_km = Use_max -
   Use_min` dependa implicitamente de um uso acumulado que só cresce;
4. a agregação por ativo descartava `Use_max` e não registrava quantas
   leituras sustentam cada linha — informação útil para julgar a confiança
   de cada ativo;
5. nunca havíamos comparado nossa agregação a uma fonte independente, embora
   `Asset_new_informations.csv` traga `Initial_km` e `nb_km` de referência
   para 80 ativos — a mesma checagem cruzada que a etapa A2 aplicou ao
   consumo;
6. a junção usava `maintenance_part_consumption_by_reason.csv` (só a média
   de consumo por motivo), quando `A2_maint_part_cons.ipynb` já produz uma
   versão mais rica (`_v2`, com soma, média, nº de eventos e nº de peças por
   motivo) — troca sugerida pela própria nota de compatibilidade de A2, sem
   exigir reescrever a lógica do `merge`.

Cada seção nova traz o diagnóstico, a decisão tomada e a justificativa —
sempre registrando o que foi feito, nunca alterando a fonte original. Esta
etapa depende das saídas de A2; execute `A2_maint_part_cons.ipynb` primeiro.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
%matplotlib inline

DATA_DIR = Path("dados_manutencao")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

usage_path = DATA_DIR / "usage_data.csv"
consumption_path = OUTPUT_DIR / "maintenance_part_consumption_by_reason_v2.csv"
failures_path = DATA_DIR / "maintenance_failures.csv"
new_info_path = DATA_DIR / "Asset_new_informations.csv"

if not consumption_path.exists():
    raise FileNotFoundError(
        "A saída da etapa A2 não foi encontrada. Execute "
        "A2_maint_part_cons.ipynb (que produz a tabela de consumo "
        "'_v2') antes desta etapa."
    )

usage_raw = pd.read_csv(usage_path)
consumption_by_reason = pd.read_csv(consumption_path)
failures = pd.read_csv(failures_path)
new_info = pd.read_csv(new_info_path)

print("Dimensões das entradas")
print(f"  uso:              {usage_raw.shape}")
print(f"  consumo (v2):     {consumption_by_reason.shape}")
print(f"  falhas:           {failures.shape}")
print(f"  info. adicionais: {new_info.shape}")

Dimensões das entradas
  uso:              (22828, 3)
  consumo (v2):     (1913, 57)
  falhas:           (1805, 2)
  info. adicionais: (80, 18)


## 1. Visão geral de `usage_data.csv`

Em `usage_data.csv`, cada ativo possui várias medições ao longo do tempo — a
sequência de medições ao longo do tempo é chamada de **série temporal**. A
chave lógica de uma leitura é `(Asset, Time)`: em princípio, cada combinação
dessas duas colunas deveria descrever uma única leitura de uso.

In [11]:
column_summary = pd.DataFrame({
    "tipo": usage_raw.dtypes.astype(str),
    "campos_vazios": usage_raw.isna().sum(),
    "valores_diferentes": usage_raw.nunique(dropna=True),
})

print("Resumo das colunas")
display(column_summary)

print(f"Eventos de uso: {len(usage_raw):,}")
print(f"Ativos distintos: {usage_raw['Asset'].nunique():,}")
print(f"Faixa de Time: {usage_raw['Time'].min()} a {usage_raw['Time'].max()} dias")

Resumo das colunas


,tipo,campos_vazios,valores_diferentes
Asset,str,0,1894
Time,int64,0,729
Use,float64,0,15554


Eventos de uso: 22,828
Ativos distintos: 1,894
Faixa de Time: 0 a 730 dias


## 2. Duplicatas exatas e leituras conflitantes — um problema de qualidade escondido

Assim como em A2, duas situações merecem atenção na chave `(Asset, Time)`:

- **duplicata exata:** a mesma linha (inclusive `Use`) aparece mais de uma
  vez — não acrescenta informação nova, é sinal de exportação ou junção
  duplicada;
- **conflito de leitura:** o mesmo `(Asset, Time)` aparece com `Use`
  **diferente** entre as linhas — aqui não sabemos qual valor é o correto,
  então a linha não deve ser descartada sem critério.

Vamos medir os dois casos antes de decidir o que fazer.

In [12]:
key_cols = ["Asset", "Time"]

exact_duplicates = usage_raw.duplicated()
print(f"Linhas totalmente duplicadas: {exact_duplicates.sum():,} "
      f"({exact_duplicates.mean():.2%} da fonte)")

group_sizes = usage_raw.groupby(key_cols).size()
print("\nDistribuição do número de linhas por chave (Asset, Time):")
print(group_sizes.value_counts().sort_index().to_string())

distinct_use = usage_raw.groupby(key_cols)["Use"].nunique()
conflicting_keys_before = distinct_use[distinct_use > 1]
print(f"\nChaves (Asset, Time) com Use conflitante: "
      f"{len(conflicting_keys_before):,} de {len(distinct_use):,}")

Linhas totalmente duplicadas: 315 (1.38% da fonte)

Distribuição do número de linhas por chave (Asset, Time):
1    20257
2     1185
3       61
4        2
5        2

Chaves (Asset, Time) com Use conflitante: 973 de 21,507


**Leitura do diagnóstico:** cerca de 1,4% das linhas são duplicatas exatas —
bem menos frequente que em A2, mas ainda um sinal de exportação duplicada
que vale remover. Os conflitos de leitura (mesma chave, `Use` diferente) são
mais comuns proporcionalmente — quase 5% das chaves — mas continuam sendo
casos em que **não sabemos qual leitura é correta**.

**Decisão desta etapa:**

- descartamos as duplicatas exatas com `drop_duplicates()`;
- **mantivemos** as leituras conflitantes — não há critério para escolher
  uma — e as sinalizamos numa coluna `reading_conflict` por linha, que será
  resumida por ativo na agregação da seção 4.

In [13]:
usage = usage_raw.drop_duplicates().copy()

conflict_flag = usage.groupby(key_cols)["Use"].transform("nunique") > 1
usage["reading_conflict"] = conflict_flag

print(f"Linhas antes da deduplicação: {len(usage_raw):,}")
print(f"Linhas depois da deduplicação: {len(usage):,} "
      f"(-{len(usage_raw) - len(usage):,})")
print(f"Linhas sinalizadas com reading_conflict=True: "
      f"{usage['reading_conflict'].sum():,}")

usage.head()

Linhas antes da deduplicação: 22,828
Linhas depois da deduplicação: 22,513 (-315)
Linhas sinalizadas com reading_conflict=True: 1,979


,Asset,Time,Use,reading_conflict
0,A403193,5,31194.652034,False
1,A403193,17,31223.536355,False
2,A403193,56,31362.706261,False
3,A403193,124,31701.440562,False
4,A403193,144,31724.489666,False


## 3. Reversões de leitura — `Use` pode diminuir ao longo do tempo?

`nb_km = Use_max - Use_min` (a fórmula de B) implicitamente supõe que `Use`
só cresce — como um odômetro. Se, para algum ativo, uma leitura posterior no
tempo for **menor** que uma leitura anterior (uma **reversão**), essa
suposição é violada: pode ser ruído de medição, dois sensores misturados sob
o mesmo `Asset`, ou uma substituição de componente que reinicia a contagem.

Como calculamos `nb_km` com o mínimo e o máximo **globais** (não uma soma de
diferenças consecutivas), reversões pequenas e isoladas não distorcem o
resultado. Mesmo assim, vale diagnosticar e sinalizar, pois uma reversão
grande indicaria um `nb_km` não confiável para aquele ativo.

In [14]:
usage_sorted = usage.sort_values(key_cols).copy()
usage_sorted["Use_prev"] = usage_sorted.groupby("Asset")["Use"].shift(1)
usage_sorted["is_reversal"] = usage_sorted["Use"] < usage_sorted["Use_prev"]

n_reversals = usage_sorted["is_reversal"].sum()
print(f"Leituras em reversão: {n_reversals} de {len(usage_sorted):,} "
      f"({n_reversals / len(usage_sorted):.3%})")

reversal_by_asset = usage_sorted.groupby("Asset")["is_reversal"].sum()
assets_with_reversal = reversal_by_asset[reversal_by_asset > 0]
print(f"Ativos com ao menos uma reversão: {len(assets_with_reversal)} "
      f"de {usage['Asset'].nunique():,}")
print(assets_with_reversal.to_string())

usage_sorted[usage_sorted["is_reversal"]][
    ["Asset", "Time", "Use", "Use_prev"]
]

Leituras em reversão: 7 de 22,513 (0.031%)
Ativos com ao menos uma reversão: 2 de 1,894
Asset
A283388    1
A333832    6


,Asset,Time,Use,Use_prev
11634,A283388,343,30127.099229,30245.262358
16019,A333832,112,28693.678368,28905.788477
16021,A333832,239,29032.704430,29408.200593
16022,A333832,329,29327.382848,29828.627920
16023,A333832,360,29431.249697,29940.080550
16027,A333832,538,30018.272447,30534.105560
16028,A333832,578,30126.515708,30560.364033


**Leitura do diagnóstico:** apenas 2 ativos (de quase 1.900) têm alguma
reversão, e as quedas são pequenas (dezenas de km, não milhares) — não há
indício de reinício de contador. `nb_km` continua confiável para o conjunto,
mas registramos a exceção.

**Decisão desta etapa:** manter as leituras e sinalizar, por ativo,
`has_use_reversal`, para que a etapa C possa tratar esses dois ativos com
mais cautela caso o modelo se mostre sensível a eles.

## 4. Das medições para uma linha por ativo — sem descartar informação

Na versão original, calculávamos `Time_min`, `Use_min` e `Use_max`, mas
descartávamos `Use_max` logo após calcular `nb_km`. Além de manter
`Use_max`, agregaremos:

- `Time_max`: último instante observado (junto com `Time_min`, define a
  janela de observação: `obs_span`);
- `n_obs`: número de leituras que sustentam a linha — ativos com poucas
  leituras merecem menos confiança;
- `n_reading_conflicts`: quantas leituras conflitantes (seção 2) o ativo
  teve;
- `has_use_reversal`: se o ativo teve alguma reversão (seção 3).

Essas quatro últimas colunas não existem na versão original de B: são sinais
de **qualidade por ativo**, não apenas de uso.

In [15]:
usage_by_asset = (
    usage
    .groupby("Asset", as_index=False)
    .agg(
        Time_min=("Time", "min"),
        Time_max=("Time", "max"),
        Use_min=("Use", "min"),
        Use_max=("Use", "max"),
        n_obs=("Time", "size"),
        n_reading_conflicts=("reading_conflict", "sum"),
    )
)
usage_by_asset["nb_km"] = (
    usage_by_asset["Use_max"] - usage_by_asset["Use_min"]
).round(2)
usage_by_asset["obs_span"] = usage_by_asset["Time_max"] - usage_by_asset["Time_min"]
usage_by_asset = usage_by_asset.merge(
    (reversal_by_asset > 0).rename("has_use_reversal"),
    left_on="Asset", right_index=True, how="left",
)

assert usage_by_asset["Asset"].is_unique
assert (usage_by_asset["nb_km"] >= 0).all()

usage_output = OUTPUT_DIR / "usage_data_by_asset_v2.csv"
usage_by_asset.to_csv(usage_output, index=False)

print(f"Tabela agregada: {usage_by_asset.shape}")
print(f"Salva em: {usage_output}")
usage_by_asset.head()

Tabela agregada: (1894, 10)
Salva em: output/usage_data_by_asset_v2.csv


,Asset,Time_min,Time_max,Use_min,Use_max,n_obs,n_reading_conflicts,nb_km,obs_span,has_use_reversal
0,A000204,46,722,31449.650982,33212.761559,15,0,1763.11,676,False
1,A000270,435,617,26378.556339,27036.768727,3,0,658.21,182,False
2,A000463,30,616,30451.537250,31894.002697,18,0,1442.47,586,False
3,A000495,6,695,30851.249561,32540.252891,16,0,1689.00,689,False
4,A001201,395,720,26136.394866,27187.900826,7,0,1051.51,325,False


## 5. Validação cruzada com `Asset_new_informations.csv`

A versão original de B nunca comparou sua agregação a uma fonte
independente. `Asset_new_informations.csv` traz, para 80 ativos,
`Initial_km` e `nb_km` calculados por algum processo externo, além de
`Time_begin_exploitation`. Isso permite checar `Use_min`, `Time_min` e
`nb_km` — a mesma ideia da validação cruzada que a etapa A2 aplicou ao
consumo.

In [16]:
overlap = new_info.merge(usage_by_asset, on="Asset", how="inner", suffixes=("_ref", "_ours"))
print(f"Ativos em comum com Asset_new_informations.csv: {len(overlap)} de {len(new_info)}")

for ref_col, our_col, label in [
    ("Initial_km", "Use_min", "Use_min vs. Initial_km"),
    ("Time_begin_exploitation", "Time_min", "Time_min vs. Time_begin_exploitation"),
    ("nb_km", "nb_km", "nb_km (nossa agregação vs. referência)"),
]:
    ref = overlap[f"{ref_col}_ref"] if f"{ref_col}_ref" in overlap else overlap[ref_col]
    ours = overlap[f"{our_col}_ours"] if f"{our_col}_ours" in overlap else overlap[our_col]
    exact_match_rate = (ref.round(2) == ours.round(2)).mean()
    mean_abs_diff = (ref - ours).abs().mean()
    correlation = ref.corr(ours)
    print(f"{label:45s} correspondência exata: {exact_match_rate:6.1%}   "
          f"diff. média: {mean_abs_diff:8.4f}   correlação: {correlation:.4f}")

Ativos em comum com Asset_new_informations.csv: 80 de 80
Use_min vs. Initial_km                        correspondência exata: 100.0%   diff. média:   0.0000   correlação: 1.0000
Time_min vs. Time_begin_exploitation          correspondência exata: 100.0%   diff. média:   0.0000   correlação: 1.0000
nb_km (nossa agregação vs. referência)        correspondência exata: 100.0%   diff. média:   0.0024   correlação: 1.0000


**Leitura do diagnóstico:** para os 80 ativos em comum, `Use_min` e
`Time_min` batem **exatamente** com `Initial_km` e
`Time_begin_exploitation` — forte evidência de que nossa agregação
reproduz corretamente o que uma fonte independente já havia calculado.
`nb_km` bate com correlação praticamente perfeita, com diferença média
residual explicada por arredondamento. Diferente do que a etapa A2 encontrou
para consumo (correspondência parcial, dependendo do motivo), aqui a
validação é essencialmente uma confirmação — o que aumenta a confiança na
fórmula `nb_km = Use_max - Use_min` para o restante dos ativos, que não têm
essa referência externa.

**Decisão desta etapa:** manter a agregação como está; a checagem não indicou
necessidade de ajuste.

## 6. Juntando as tabelas (agora com o consumo mais rico da etapa A2)

A tabela de referência continua sendo a de uso (queremos manter todos os
ativos observados em operação), por meio de um **left join**. A diferença
em relação à versão original é a fonte de consumo:
`maintenance_part_consumption_by_reason_v2.csv` traz, por motivo, quatro
métricas (`qty_sum`, `qty_mean`, `n_events`, `n_parts`) em vez de apenas a
média — 56 colunas de consumo em vez de 14. A lógica do `merge` não muda:
continuamos validando a cardinalidade **um para um** antes de juntar.

In [17]:
for name, table in {
    "uso resumido": usage_by_asset,
    "consumo por motivo (v2)": consumption_by_reason,
    "situação de falha": failures,
}.items():
    duplicated = table["Asset"].duplicated().sum()
    print(f"{name:24s}: {len(table):4d} linhas; "
          f"{duplicated} ativos repetidos")
    assert duplicated == 0

integrated = usage_by_asset.merge(
    consumption_by_reason,
    on="Asset",
    how="left",
    validate="one_to_one",
    indicator="consumption_match",
)

print("\nAtivos encontrados na tabela de consumo")
print(integrated["consumption_match"].value_counts().to_string())

integrated = integrated.merge(
    failures,
    on="Asset",
    how="left",
    validate="one_to_one",
    indicator="failure_match",
)

print("\nAtivos encontrados na tabela de falhas")
print(integrated["failure_match"].value_counts().to_string())

uso resumido            : 1894 linhas; 0 ativos repetidos
consumo por motivo (v2) : 1913 linhas; 0 ativos repetidos
situação de falha       : 1805 linhas; 0 ativos repetidos

Ativos encontrados na tabela de consumo
consumption_match
both          1893
left_only        1
right_only       0

Ativos encontrados na tabela de falhas
failure_match
both          1805
left_only       89
right_only       0


## 7. Verificando o resultado da junção

As mesmas ressalvas da versão original continuam válidas: campo vazio após a
junção não significa necessariamente erro, e `failure_bin` continua sendo o
**alvo**. Com 56 colunas de consumo em vez de 14, faz ainda mais sentido
olhar o topo do ranking de faltantes.

In [18]:
asset_modelization_table = (
    integrated
    .drop(columns=["consumption_match", "failure_match"])
)

missing_summary = pd.DataFrame({
    "faltantes": asset_modelization_table.isna().sum(),
    "percentual": asset_modelization_table.isna().mean(),
}).sort_values("percentual", ascending=False)

print(f"Tabela integrada: {asset_modelization_table.shape}")
print(f"Ativos únicos: {asset_modelization_table['Asset'].is_unique}")
print(f"Ativos sinalizados com n_reading_conflicts > 0: "
      f"{(asset_modelization_table['n_reading_conflicts'] > 0).sum()}")
print(f"Ativos sinalizados com has_use_reversal: "
      f"{asset_modelization_table['has_use_reversal'].sum()}")
print("\nColunas com maior percentual de faltantes")
display(missing_summary.head(15).style.format({"percentual": "{:.1%}"}))

model_output = OUTPUT_DIR / "asset_modelization_table_v2.csv"
asset_modelization_table.to_csv(model_output, index=False)

print(f"Salva em: {model_output}")
asset_modelization_table.head()

Tabela integrada: (1894, 67)
Ativos únicos: True
Ativos sinalizados com n_reading_conflicts > 0: 718
Ativos sinalizados com has_use_reversal: 2

Colunas com maior percentual de faltantes


,faltantes,percentual
R782_qty_sum,1893,99.9%
R782_n_events,1893,99.9%
R782_n_parts,1893,99.9%
R396_qty_sum,1893,99.9%
R396_qty_mean,1893,99.9%
R396_n_events,1893,99.9%
R782_qty_mean,1893,99.9%
R396_n_parts,1893,99.9%
R064_qty_sum,1892,99.9%
R064_n_events,1892,99.9%


Salva em: output/asset_modelization_table_v2.csv


,Asset,Time_min,Time_max,Use_min,Use_max,n_obs,n_reading_conflicts,nb_km,obs_span,has_use_reversal,...,R396_n_parts,R417_n_parts,R446_n_parts,R565_n_parts,R575_n_parts,R606_n_parts,R707_n_parts,R782_n_parts,R783_n_parts,failure_bin
0,A000204,46,722,31449.650982,33212.761559,15,0,1763.11,676,False,...,NaN,NaN,3.0,26.0,NaN,NaN,41.0,NaN,NaN,1.0
1,A000270,435,617,26378.556339,27036.768727,3,0,658.21,182,False,...,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN
2,A000463,30,616,30451.537250,31894.002697,18,0,1442.47,586,False,...,NaN,NaN,5.0,NaN,NaN,NaN,48.0,NaN,4.0,1.0
3,A000495,6,695,30851.249561,32540.252891,16,0,1689.00,689,False,...,NaN,NaN,20.0,36.0,NaN,NaN,36.0,NaN,NaN,0.0
4,A001201,395,720,26136.394866,27187.900826,7,0,1051.51,325,False,...,NaN,NaN,NaN,NaN,NaN,NaN,21.0,NaN,NaN,0.0


## Ponto de verificação da etapa B2

```text
uso por evento ── dedup + flags ── resumo ──────┐
                                                 ├── junção ── tabela por ativo
consumo por motivo (v2, A2) ─────────────────────┤
falha por ativo ──────────────────────────────────┘
```

- **Arquivos criados:** `usage_data_by_asset_v2.csv` e
  `asset_modelization_table_v2.csv` (não sobrescrevem as saídas de B, que
  continuam disponíveis para comparação);
- **Coluna de ligação:** `Asset`, com um único valor por linha;
- **O que cada linha representa ao final:** um ativo;
- **Transformações aplicadas, em ordem, além da versão original:**
  1. remoção de duplicatas exatas em `usage_data.csv`;
  2. sinalização de leituras conflitantes (`reading_conflict` →
     `n_reading_conflicts` por ativo);
  3. diagnóstico de reversões de leitura (`has_use_reversal` por ativo);
  4. agregação mais completa (`Use_max`, `Time_max`, `n_obs`, `obs_span`
     preservados, não descartados);
  5. validação cruzada de `Use_min`, `Time_min` e `nb_km` com
     `Asset_new_informations.csv`;
  6. junção com a tabela de consumo mais rica da etapa A2 (`_v2`, 56 colunas
     em vez de 14).

O **inner join** continua sendo outro tipo de junção possível: ele conserva
somente os ativos que aparecem nas duas tabelas. **Discussão:** com
`n_reading_conflicts` e `has_use_reversal` agora disponíveis como colunas,
como você usaria esses sinais de qualidade na etapa C — como filtro, como
peso, ou como mais duas variáveis de entrada do modelo?